In [5]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
# Step 1: Define the structure of the Bayesian Network
model = DiscreteBayesianNetwork([
    ('StudyHours', 'Grade'),
    ('Intelligence', 'Grade'),
    ('Difficulty', 'Grade'),
    ('Grade', 'Pass')
])

cpd_StudyHours = TabularCPD(variable='StudyHours', variable_card=2,
values=[[0.6], [0.4]])

cpd_Intelligence = TabularCPD(variable='Intelligence', variable_card=2,
values=[[0.7], [0.3]])

cpd_Difficulty = TabularCPD(variable='Difficulty', variable_card=2,
values=[[0.4], [0.6]])

cpd_Grade = TabularCPD(
    variable='Grade',
    variable_card=3,
    values = [
        [0.7, 0.85, 0.3, 0.35, 0.6, 0.65, 0.15, 0.1], #A
        [0.25, 0.1, 0.45, 0.5, 0.2, 0.25, 0.3, 0.3], #B
        [0.05, 0.05, 0.25, 0.15, 0.2, 0.1, 0.55, 0.6]  #C=
    ],
    evidence=['StudyHours', 'Intelligence', 'Difficulty'],
    evidence_card=[2, 2, 2]
    )

cpd_Pass = TabularCPD(
    variable='Pass',
    variable_card=2,
    values = [
        [0.95, 0.8, 0.5], #pass
        [0.05, 0.2, 0.5]   #fail
    ],
    evidence=['Grade'],
    evidence_card=[3]
    )

model.add_cpds(cpd_StudyHours, cpd_Intelligence, cpd_Difficulty, cpd_Grade, cpd_Pass)

assert model.check_model(), "model is incorrect"
infer = VariableElimination(model)

result = infer.query(variables=['Pass'], evidence={'StudyHours': 0, 'Difficulty': 0})
print(result)

+---------+-------------+
| Pass    |   phi(Pass) |
+=========+=============+
| Pass(0) |      0.8540 |
+---------+-------------+
| Pass(1) |      0.1460 |
+---------+-------------+


In [6]:
import numpy as np

states = ["Sunny", "Cloudy", "Rainy"]
transition_matrix = np.array([
    [0.6, 0.3, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.3, 0.5]
])

def simulate_markov_process(initial_state, num_steps):
    current_state = initial_state
    state_sequence = [current_state]
    for _ in range(num_steps - 1):
        idx = states.index(current_state)
        next_state = np.random.choice(states, p=transition_matrix[idx])
        state_sequence.append(next_state)
        current_state = next_state
    return state_sequence

initial_state = "Sunny"
num_steps = 10
state_sequence = simulate_markov_process(initial_state, num_steps)

print(f"Sequence: {' -> '.join(state_sequence)}")

trials = 10000
rainy_count = 0
for _ in range(trials):
    seq = simulate_markov_process(initial_state, num_steps)
    if seq.count("Rainy") >= 3:
        rainy_count += 1

print(f"Prob of >= 3 rainy days: {rainy_count / trials}")

Sequence: Sunny -> Rainy -> Cloudy -> Cloudy -> Rainy -> Rainy -> Cloudy -> Cloudy -> Cloudy -> Cloudy
Prob of >= 3 rainy days: 0.3809
